# Decagon CGR → EMPR → SVM pipeline

A compact pipeline for loading diploid mTOR sequences, generating CGR/EMPR features, tuning an RBF-SVM, and evaluating balanced and 1:4 imbalanced scenarios.

Labels: **0 = control**, **1 = patient**.

In [1]:
print('hello')

hello


In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

DATA_PATH = Path("data_and_cache/mTOR_data.pkl")
FEATURE_CACHE = Path("data_and_cachend_cache/mTOR_empr_features_1431.pkl")
RANDOM_SEED = 42
S_VALUES = [10, 20, 30, 40, 50]
N_RUNS = 10  # Increase to 100 for the final experiment.

PARAM_GRID = {
    "svm__C": [10.0**i for i in range(-4, 5)],
    "svm__gamma": [10.0**i for i in range(-4, 5)],
}

## 1. Load diploid sequence data

The input pickle must contain `controls`, `patients`, and `gene_names`.

In [3]:
with DATA_PATH.open("rb") as file:
    sequence_data = pickle.load(file)

required_keys = {"controls", "patients", "gene_names"}
missing_keys = required_keys - sequence_data.keys()
if missing_keys:
    raise KeyError(f"Missing keys in {DATA_PATH}: {sorted(missing_keys)}")

controls = sequence_data["controls"]
patients = sequence_data["patients"]
gene_names = sequence_data["gene_names"]

print(f"Controls: {len(controls)}, patients: {len(patients)}, genes: {len(gene_names)}")

Controls: 400, patients: 400, genes: 31


## 2. Decagon CGR and EMPR feature generation

Each gene sequence becomes a decagon CGR matrix. The matrices of one subject are stacked into a tensor, from which the EMPR vector is calculated.

In [4]:
IUPAC_DECAGON = {
    "A": (1.000, 0.000), "R": (0.809, 0.588),
    "M": (0.309, 0.951), "G": (-0.309, 0.951),
    "S": (-0.809, 0.588), "C": (-1.000, 0.000),
    "Y": (-0.809, -0.588), "K": (-0.309, -0.951),
    "T": (0.309, -0.951), "W": (0.809, -0.588),
}


def create_decagon_cgr(sequence, scale=700, ratio=0.809):
    """Convert one IUPAC sequence into a decagon CGR frequency matrix."""
    targets = {base: np.asarray(point) for base, point in IUPAC_DECAGON.items()}
    matrix = np.zeros((scale, scale), dtype=np.float64)
    position = np.zeros(2, dtype=np.float64)

    for base in sequence:
        target = targets.get(base)
        if target is None:
            continue
        position += ratio * (target - position)
        indices = ((position + 1) * (scale - 1) / 2).astype(int)
        x, y = np.clip(indices, 0, scale - 1)
        matrix[x, y] += 1

    return matrix


def calculate_empr_features(tensor):
    """Return the concatenated EMPR mode features of one subject tensor."""
    n1, n2, n3 = tensor.shape
    eps = 1e-10
    w1, w2, w3 = np.ones(n1) / n1, np.ones(n2) / n2, np.ones(n3) / n3
    total = tensor.sum() + eps

    s1 = np.tensordot(tensor, np.outer(w2, w3), axes=([1, 2], [0, 1])) / total
    s2 = np.tensordot(tensor, np.outer(w1, w3), axes=([0, 2], [0, 1])) / total
    s3 = np.tensordot(tensor, np.outer(w1, w2), axes=([0, 1], [0, 1])) / total
    s1 /= s1.sum() + eps
    s2 /= s2.sum() + eps
    s3 /= s3.sum() + eps

    weights = np.einsum("i,j,k->ijk", w1, w2, w3)
    scores = np.einsum("i,j,k->ijk", s1, s2, s3)
    g0 = np.sum(tensor * weights * scores)

    g1 = np.tensordot(tensor, np.outer(w2, w3) * np.outer(s2, s3), axes=([1, 2], [0, 1])) - g0 * s1
    g2 = np.tensordot(tensor, np.outer(w1, w3) * np.outer(s1, s3), axes=([0, 2], [0, 1])) - g0 * s2
    g3 = np.tensordot(tensor, np.outer(w1, w2) * np.outer(s1, s2), axes=([0, 1], [0, 1])) - g0 * s3
    return np.concatenate((g1, g2, g3))


def extract_population_features(subjects, n_genes, label, scale=700):
    features = []
    for index, subject in enumerate(subjects, start=1):
        gene_matrices = [create_decagon_cgr(subject[g], scale=scale) for g in range(n_genes)]
        features.append(calculate_empr_features(np.stack(gene_matrices, axis=2)))
        if index % 25 == 0 or index == len(subjects):
            print(f"Label {label}: {index}/{len(subjects)} subjects")
    return features

In [5]:
if FEATURE_CACHE.exists():
    with FEATURE_CACHE.open("rb") as file:
        cached = pickle.load(file)
    X, y = np.asarray(cached["X"]), np.asarray(cached["y"])
    print(f"Loaded cached features: X={X.shape}, y={y.shape}")
else:
    n_genes = len(gene_names)
    control_features = extract_population_features(controls, n_genes, label=0)
    patient_features = extract_population_features(patients, n_genes, label=1)

    X = np.asarray(control_features + patient_features)
    y = np.concatenate((np.zeros(len(controls), dtype=int), np.ones(len(patients), dtype=int)))

    FEATURE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    with FEATURE_CACHE.open("wb") as file:
        pickle.dump({"X": X, "y": y}, file, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Generated and cached features: X={X.shape}, y={y.shape}")

if len(X) != len(y) or set(np.unique(y)) != {0, 1}:
    raise ValueError("X and y must contain aligned samples from both binary classes.")

Loaded cached features: X=(800, 1431), y=(800,)


## 3. SVM tuning and evaluation

Scaling is part of the scikit-learn pipeline, so every CV fold learns its scaler only from its own training data. Hyperparameters are selected by F1 score.

In [6]:
def tune_svm(X_train, y_train, class_weight=None, seed=RANDOM_SEED):
    model = Pipeline([
        ("scale", MinMaxScaler(feature_range=(-1, 1))),
        ("svm", SVC(kernel="rbf", class_weight=class_weight)),
    ])
    cv_folds = min(5, int(np.bincount(y_train).min()))
    if cv_folds < 2:
        raise ValueError("Each training class needs at least two samples.")

    search = GridSearchCV(
        model,
        PARAM_GRID,
        scoring="f1",
        cv=StratifiedKFold(cv_folds, shuffle=True, random_state=seed),
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    return search


def evaluate_model(model, X_test, y_test):
    prediction = model.predict(X_test)
    score = model.decision_function(X_test)
    return {
        "OA (%)": 100 * accuracy_score(y_test, prediction),
        "Balanced accuracy": balanced_accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "AUC": roc_auc_score(y_test, score),
    }


def summarize_runs(records, group="S"):
    frame = pd.DataFrame(records)
    metrics = ["OA (%)", "Balanced accuracy", "Precision", "Recall", "F1", "AUC"]
    return frame.groupby(group)[metrics].agg(["mean", "std"]).round(4)

### Balanced scenario

For each run, `S` controls and `S` patients form the training set; all remaining samples form the test set.

In [7]:
def run_balanced_experiments(X, y, s_values=S_VALUES, n_runs=N_RUNS, seed=RANDOM_SEED):
    class_indices = {label: np.flatnonzero(y == label) for label in (0, 1)}
    records = []

    for s in s_values:
        if s >= min(map(len, class_indices.values())):
            raise ValueError(f"S={s} leaves no test samples in at least one class.")
        rng = np.random.default_rng(seed + s)

        for run in range(n_runs):
            train_indices = np.concatenate([
                rng.choice(class_indices[0], s, replace=False),
                rng.choice(class_indices[1], s, replace=False),
            ])
            test_indices = np.setdiff1d(np.arange(len(y)), train_indices)

            search = tune_svm(X[train_indices], y[train_indices], seed=seed + run)
            metrics = evaluate_model(search.best_estimator_, X[test_indices], y[test_indices])
            records.append({"S": s, "Run": run + 1, **metrics, **search.best_params_})

        print(f"Balanced S={s}: {n_runs} runs completed")

    return pd.DataFrame(records)


balanced_runs = run_balanced_experiments(X, y)
balanced_summary = summarize_runs(balanced_runs)
balanced_summary

Balanced S=10: 10 runs completed
Balanced S=20: 10 runs completed
Balanced S=30: 10 runs completed
Balanced S=40: 10 runs completed
Balanced S=50: 10 runs completed


OA (%)         Balanced accuracy         Precision          Recall  \
       mean     std              mean     std      mean     std    mean   
S                                                                         
10  83.1923  7.9219            0.8319  0.0792    0.7702  0.0880  0.9662   
20  95.0526  1.7926            0.9505  0.0179    0.9250  0.0375  0.9826   
30  96.6351  1.5636            0.9664  0.0156    0.9504  0.0330  0.9857   
40  97.4306  1.2392            0.9743  0.0124    0.9580  0.0240  0.9928   
50  97.4714  1.2689            0.9747  0.0127    0.9609  0.0227  0.9903   

                F1             AUC          
       std    mean     std    mean     std  
S                                           
10  0.0272  0.8549  0.0602  0.9324  0.0654  
20  0.0182  0.9524  0.0164  0.9937  0.0039  
30  0.0201  0.9672  0.0146  0.9973  0.0020  
40  0.0027  0.9749  0.0118  0.9984  0.0014  
50  0.0065  0.9752  0.0121  0.9978  0.0018

### Imbalanced 1:4 scenario

The evaluation pool contains 100 patients and 400 controls. Each run trains on `S` patients and `4S` controls. Class weighting prevents the SVM from collapsing to the majority class.

In [8]:
def make_imbalanced_pool(X, y, n_patients=100, seed=RANDOM_SEED):
    control_indices = np.flatnonzero(y == 0)
    patient_indices = np.flatnonzero(y == 1)
    if len(control_indices) < 4 * n_patients or len(patient_indices) < n_patients:
        raise ValueError("Not enough samples to construct the requested 1:4 pool.")

    rng = np.random.default_rng(seed)
    selected = np.concatenate([
        control_indices,
        rng.choice(patient_indices, n_patients, replace=False),
    ])
    return X[selected], y[selected]


def run_imbalanced_experiments(X, y, s_values=S_VALUES, n_runs=N_RUNS, seed=RANDOM_SEED):
    pool_X, pool_y = make_imbalanced_pool(X, y, seed=seed)
    class_indices = {label: np.flatnonzero(pool_y == label) for label in (0, 1)}
    records = []

    for s in s_values:
        if s >= len(class_indices[1]) or 4 * s >= len(class_indices[0]):
            raise ValueError(f"S={s} leaves no test samples in at least one class.")
        rng = np.random.default_rng(seed + s)

        for run in range(n_runs):
            train_indices = np.concatenate([
                rng.choice(class_indices[0], 4 * s, replace=False),
                rng.choice(class_indices[1], s, replace=False),
            ])
            test_indices = np.setdiff1d(np.arange(len(pool_y)), train_indices)

            search = tune_svm(
                pool_X[train_indices], pool_y[train_indices],
                class_weight="balanced", seed=seed + run,
            )
            metrics = evaluate_model(search.best_estimator_, pool_X[test_indices], pool_y[test_indices])
            records.append({"S": s, "Run": run + 1, **metrics, **search.best_params_})

        print(f"Imbalanced S={s} ({s} patients, {4 * s} controls): {n_runs} runs completed")

    return pd.DataFrame(records)


imbalanced_runs = run_imbalanced_experiments(X, y)
imbalanced_summary = summarize_runs(imbalanced_runs)
imbalanced_summary

Imbalanced S=10 (10 patients, 40 controls): 10 runs completed
Imbalanced S=20 (20 patients, 80 controls): 10 runs completed
Imbalanced S=30 (30 patients, 120 controls): 10 runs completed
Imbalanced S=40 (40 patients, 160 controls): 10 runs completed
Imbalanced S=50 (50 patients, 200 controls): 10 runs completed


OA (%)         Balanced accuracy         Precision          Recall  \
       mean     std              mean     std      mean     std    mean   
S                                                                         
10  95.6000  2.5805            0.9262  0.0380    0.9192  0.1072  0.8767   
20  97.5750  1.2531            0.9680  0.0204    0.9316  0.0635  0.9550   
30  97.6571  1.4241            0.9779  0.0089    0.9136  0.0612  0.9800   
40  98.1333  0.9054            0.9827  0.0087    0.9278  0.0385  0.9850   
50  98.8000  0.5963            0.9858  0.0078    0.9596  0.0250  0.9820   

                F1             AUC          
       std    mean     std    mean     std  
S                                           
10  0.0874  0.8901  0.0579  0.9922  0.0072  
20  0.0476  0.9408  0.0292  0.9986  0.0011  
30  0.0074  0.9446  0.0322  0.9988  0.0011  
40  0.0146  0.9551  0.0209  0.9991  0.0008  
50  0.0148  0.9705  0.0145  0.9996  0.0003